# 01 · Quickstart: scoring pairs with CisLink-E2G and CisLink-V2G

CisLink has two models, both L2-regularized logistic regressions trained once on 10,250 K562 CRISPR-tested element–gene pairs:

| Model | Question | Inputs |
|---|---|---|
| **CisLink-E2G** (37 features) | Which elements regulate which genes in this cell? | accessibility, 11 histone marks, CTCF, expression, the cell's own Hi-C, distance and gene context |
| **CisLink-V2G** (13 features) | Which gene does this GWAS variant regulate? | accessibility, expression, multi-tissue Hi-C megamap, distance and gene context; every variant scored, in a peak or not |

Both take a table of **harmonized** features (one row per pair; see `docs/features.md`). This notebook scores the example table.

In [1]:
import pandas as pd

from cislink import load_model, score

e2g, v2g = load_model("e2g"), load_model("v2g")
print(e2g.name, len(e2g.features), "features;", v2g.name, len(v2g.features), "features")
x = pd.read_csv("../examples/example_features.tsv", sep="\t")
x.iloc[:3, :8]

CisLink-E2G 37 features; CisLink-V2G 13 features


,ID,Enh,Prom,TX,Tx_expression,Gene_expression,DNase_Enh,DNase_Prom
0,1:chr1:11858311:11858930:chr1:11796141:1179614...,chr1:11858311:11858930,chr1:11795891:11796391,ENST00000376637.7,0.446112,42.414007,73715.049,17079.322
1,0:chr1:11858311:11858930:chr1:11751707:1175170...,chr1:11858311:11858930,chr1:11740918:11741418,ENST00000445656.5,1.021741,120.897580,73715.049,14354.951
2,0:chr1:11858311:11858930:chr1:11120081:1112008...,chr1:11858311:11858930,chr1:11115011:11115511,ENST00000475189.1,0.935396,45.019978,73715.049,2539.415


In [2]:
s = score(x)
s[["ID", "CisLink-E2G", "CisLink-V2G"]].head(10)

,ID,CisLink-E2G,CisLink-V2G
0,1:chr1:11858311:11858930:chr1:11796141:1179614...,0.629848,0.304891
1,0:chr1:11858311:11858930:chr1:11751707:1175170...,0.102580,0.029263
2,0:chr1:11858311:11858930:chr1:11120081:1112008...,0.060560,0.012055
3,0:chr1:11942780:11943706:chr1:11120081:1112008...,0.013544,0.002566
4,0:chr1:12043399:12043899:chr1:11751707:1175170...,0.001444,0.004271
5,0:chr1:12043399:12043899:chr1:11120081:1112008...,0.002977,0.004106
6,0:chr1:12065199:12066325:chr1:11751707:1175170...,0.001255,0.001293
7,0:chr1:12065199:12066325:chr1:11120081:1112008...,0.006320,0.001400
8,0:chr1:12098264:12099490:chr1:11796141:1179614...,0.003753,0.004976
9,0:chr1:12098264:12099490:chr1:11751707:1175170...,0.000287,0.000821


## Reading the scores

Scores are probabilities on the scale of the K562 CRISPR training pairs (4.6% positive), not absolute probabilities in a new cell. Rank candidates by score, or compare with the **recall-based thresholds**: the score at which held-out K562 pairs reach 70% or 50% recall.

In [3]:
for m in (e2g, v2g):
    print(f"{m.name}: 70% recall threshold {m.threshold(70):.3f}; 50% recall threshold {m.threshold(50):.3f}")
s["E2G above 70% threshold"] = s["CisLink-E2G"] >= e2g.threshold(70)
s["E2G above 70% threshold"].value_counts()

CisLink-E2G: 70% recall threshold 0.180; 50% recall threshold 0.529
CisLink-V2G: 70% recall threshold 0.139; 50% recall threshold 0.410


E2G above 70% threshold
False    21
True      4
Name: count, dtype: int64

## Why a pair scores high: per-feature contributions

For a logistic regression, each feature's contribution to the log-odds is its coefficient times its standardized (transformed) value; the contributions sum to the log-odds minus the intercept.

In [4]:
top = s["CisLink-E2G"].idxmax()
c = e2g.contributions(x.loc[[top]]).T.iloc[:, 0].sort_values(key=abs, ascending=False)
print(x.loc[top, "ID"], "score", round(s.loc[top, "CisLink-E2G"], 3))
c.head(10).round(3)

1:chr1:11858311:11858930:chr1:11796141:11796142:AGTRAP|K562|Xie score 0.63


H3K27ac_Enh        2.402
DNase_Enh          1.056
n_genes_between    1.027
H3K9ac_Enh        -0.606
log10_distance     0.571
H3K4me3_Enh        0.537
ubiquitous_gene    0.344
H3K4me2_Enh       -0.283
H3K4me1_Enh        0.245
distance           0.224
Name: 0, dtype: float64

## Command line

```bash
cislink info
cislink score examples/example_features.tsv -o scores.tsv --keep ID --threshold 70
```

In [5]:
import subprocess

cmd = ["cislink", "score", "../examples/example_features.tsv", "--keep", "ID", "--threshold", "70"]
print(subprocess.run(cmd, capture_output=True, text=True).stdout[:600])

ID	CisLink-E2G	CisLink-E2G >= 70% recall threshold	CisLink-V2G	CisLink-V2G >= 70% recall threshold
1:chr1:11858311:11858930:chr1:11796141:11796142:AGTRAP|K562|Xie	0.629848	True	0.304891	True
0:chr1:11858311:11858930:chr1:11751707:11751708:MAD2L2|K562|Xie	0.10258	False	0.0292628	False
0:chr1:11858311:11858930:chr1:11120081:11120082:SRM|K562|Xie	0.0605601	False	0.0120552	False
0:chr1:11942780:11943706:chr1:11120081:11120082:SRM|K562|Xie	0.0135444	False	0.00256553	False
0:chr1:12043399:12043899:chr1:11751707:11751708:MAD2L2|K562|Morris	0.00144357	False	0.00427081	False
0:chr1:12043399:12043899:ch
